# Chapter 4 - Pipeline test and eval

In [14]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import json
#===
import sys
from pathlib import Path

In [15]:
project_root = Path().resolve().parent.parent
sys.path.append(str(project_root))
project_root

PosixPath('/Users/alejandrofp/Desktop/Projects/03_Flagship_Portfolio/job-intelligence-engine')

In [16]:
from src.job_intel.pipelines.chapter4_recommender import run_recommender_pipeline
from src.job_intel.v2_updates.features.career_simulator import (
    SimulationScenario,
    SimulationConfig,
)

## Run pipeline from demo

In [ ]:
import os
os.getcwd()

# 1) load json -> dict
with open("/Users/alejandrofp/Desktop/Projects/03_Flagship_Portfolio/job-intelligence-engine/src/job_intel/evaluation/recommender_demo.json", "r") as f:
    cfg = json.load(f)

# 2) unpack blocks
user_inputs = cfg["user_inputs"]
pipeline_params = cfg.get("pipeline_params", {})
sim_block = cfg.get("career_simulation", {})

# 3) build sim objects
run_career_sim = bool(sim_block.get("run_career_sim", False))
scenarios = [SimulationScenario(**d) for d in sim_block.get("scenarios", [])]
config = SimulationConfig(**sim_block.get("config", {})) if sim_block.get("config") else SimulationConfig()

# 4) run pipeline
recommender, explanation, upskill, sim = run_recommender_pipeline(
    **user_inputs,
    **pipeline_params,
    run_career_sim=run_career_sim,
    scenarios=scenarios,
    config=config,
)

INFO | src.job_intel.pipelines.chapter4_recommender | Running Chapter 4 recommender...
INFO | src.job_intel.pipelines.chapter4_recommender | Building explanations (tau=0.50)...
INFO | src.job_intel.pipelines.chapter4_recommender | Running upskilling recommender (top_n_skills=3)...


Predicting salary based on input skills...
Acceptance shape test passed: True
Acceptance alignment test passed: True
Applying the suitability index threshold 0.7...
* Returning jobs after applying s_min=0.7.
* Suitable jobs identified = 287.
* 1015 filtered out due to low suitability.
Applying competitiveness filter into 2 buckets: "best_now" and "stretch".
Number of "Best-now" jobs = 274
Number of "Stretch" jobs = 13
Computing ranking score based on suitability and competitiveness.

Best-now salary comparison (Top-N):
---------------------------
Expected mean salary: 94350.00
Predicted mean salary (based on your skills): 102217.39
Delta (expected - predicted): -7867.39

Stretch salary comparison (Top-N):
---------------------------
Expected mean salary: 207500.00
Predicted mean salary (based on your skills): 130425.23
Delta (expected - predicted): 77074.77

Mean salary jump from "Best-now" to "Stretch" jobs (Top-N): 84942.16


INFO | src.job_intel.pipelines.chapter4_recommender | Running career simulation (n_scenarios=1)...


Upskilling report
-----------------

Ranking logic:
- Universe is frozen by job_id (candidate_override_df), so deltas are comparable.
- Missing families come from explained stretch jobs (missing_families).
- Each scenario injects representative tokens for one family into skill_text.
- Scenarios are skipped if the injected tokens do not change the extracted user skill_vector.
- Deltas are percentage points (bounded 0–1 metrics × 100).
- Ranking rewards promotions + score gains (esp. baseline-stretch), penalises demotions + worst-tail harms.
- Guardrail: demotion_rate <= 0.0.

Top 3 recommended skill families (with example tokens):
* cloud__advanced: ['gpu', 'parallel computing']
* analytics_stats__intermediate: ['statistical analysis', 'marketing analytics', 'customer analytics']
* core_programming__advanced: ['multithreading']

Top scenarios summary:
                                       upskill_impact_score  promotion_rate  \
upskill_scenario                                          

In [20]:
explanation['tables']['top_stretch_explained']

,job_id,Size,Sector,Industry,state,title_rich,sal_mean,pred_sal,suitability,competitiveness_index,...,bucket,why_bucket,why_rank,salary_context,skill_match_norm,expected_missing_norm,missing_families,covered_families,n_missing_families,n_covered_families
0,2992,10000+ employees,Information Technology,Internet,CA,ML_AI_data_scientist,225000.0,147836.468750,0.948958,0.502968,...,stretch,Barrier is higher (competitiveness 0.50 > 0.50),Ranked by score = 0.95 - (0.50 * 0.50) = 0.70,Market mean = 225000.00; predicted for you = 1...,0.927082,0.011697,"[data_engineering_pipelines__intermediate, pro...","[core_programming__basic, ml_ai__basic, ml_ai_...",2,9
1,169,10000+ employees,Insurance,Insurance Agencies & Brokerages,NY,general_data_data_scientist,205000.0,129874.109375,0.883632,0.513319,...,stretch,Barrier is higher (competitiveness 0.51 > 0.50),Ranked by score = 0.88 - (0.50 * 0.51) = 0.63,Market mean = 205000.00; predicted for you = 1...,0.833760,0.046607,"[data_engineering_pipelines__intermediate, db_...","[core_programming__basic, ml_ai__basic, ml_ai_...",4,9
2,193,1 to 50 employees,Unknown,Unknown,NY,general_data_data_scientist,205000.0,125704.921875,0.869911,0.510367,...,stretch,Barrier is higher (competitiveness 0.51 > 0.50),Ranked by score = 0.87 - (0.50 * 0.51) = 0.61,Market mean = 205000.00; predicted for you = 1...,0.814159,0.040703,"[cloud__advanced, db_storage__intermediate, so...","[core_programming__basic, ml_ai__basic, ml_ai_...",4,7
3,1205,10000+ employees,Manufacturing,Chemical Manufacturing,TX,general_data_data_scientist,177500.0,104465.515625,0.857235,0.506442,...,stretch,Barrier is higher (competitiveness 0.51 > 0.50),Ranked by score = 0.86 - (0.50 * 0.51) = 0.60,Market mean = 177500.00; predicted for you = 1...,0.844264,0.066263,"[core_programming__intermediate, data_engineer...","[core_programming__basic, ml_ai__basic, ml_ai_...",7,10
4,3004,10000+ employees,Information Technology,Computer Hardware & Software,CA,general_data_data_scientist,225000.0,144245.140625,0.784247,0.506484,...,stretch,Barrier is higher (competitiveness 0.51 > 0.50),Ranked by score = 0.78 - (0.50 * 0.51) = 0.53,Market mean = 225000.00; predicted for you = 1...,0.691781,0.018728,"[core_programming__intermediate, data_engineer...","[core_programming__basic, ml_ai__basic, ml_ai_...",5,7


## Baseline recommender output contracts

In [11]:
report = {}

# Stuff exist in recommender
if isinstance(recommender['tables'], dict):
    report['Recommender_tables'] = 'PASS'
else:
    report['Recommender_tables'] = 'FAIL'

if isinstance(recommender['tables']['scored_universe'], pd.DataFrame):
    report['Recommender_tables_universe'] = 'PASS'
else:
    report['Recommender_tables_universe'] = 'FAIL'

if isinstance(recommender['tables']['top_best_now'], pd.DataFrame):
    report['Recommender_tables_best_now'] = 'PASS'
else:
    report['Recommender_tables_best_now'] = 'FAIL'

if isinstance(recommender['tables']['top_stretch'], pd.DataFrame):
    report['Recommender_tables_stretch'] = 'PASS'
else:
    report['Recommender_tables_stretch'] = 'FAIL'

# Required cols

required_cols = ['job_id', 'score', 'bucket', 'suitability', 'competitiveness_index']

def _require_cols(df: pd.DataFrame, cols: list[str], *, where: str) -> None:
    missing = [c for c in cols if c not in df.columns]
    if missing:
        return False
    else:
        return True

if _require_cols(recommender['tables']['top_stretch'].reset_index(), required_cols, where = 'top_strech'):
    report['Recommender_tables_stretch_cols'] = 'PASS'
else:
    report['Recommender_tables_stretch_cols'] = 'FAIL'

if _require_cols(recommender['tables']['top_best_now'].reset_index(), required_cols, where = 'top_best_now'):
    report['Recommender_tables_best_now_cols'] = 'PASS'
else:
    report['Recommender_tables_best_now_cols'] = 'FAIL'

# Basic sanity

if recommender['tables']['top_stretch'].reset_index()['job_id'].isna().any():
    report['Recommender_tables_stretch_index'] = 'FAIL'
else:
    report['Recommender_tables_stretch_index'] = 'PASS'

if recommender['tables']['top_best_now'].reset_index()['job_id'].isna().any():
    report['Recommender_tables_best_now_index'] = 'FAIL'
else:
    report['Recommender_tables_best_now_index'] = 'PASS'

if recommender['tables']['top_stretch']['score'].isnull().any():
    report['Recommender_tables_stretch_score'] = 'FAIL'
else:
    report['Recommender_tables_stretch_score'] = 'PASS'

if recommender['tables']['top_best_now']['score'].isnull().any():
    report['Recommender_tables_best_now_score'] = 'FAIL'
else:
    report['Recommender_tables_best_now_score'] = 'PASS'

if recommender['tables']['top_stretch']['bucket'].any() == 'best_now':
    report['Recommender_tables_stretch_bucket'] = 'FAIL'
else:
    report['Recommender_tables_stretch_bucket'] = 'PASS'

if recommender['tables']['top_best_now']['bucket'].any() == 'stretch':
    report['Recommender_tables_best_now_bucket'] = 'FAIL'
else:
    report['Recommender_tables_best_now_bucket'] = 'PASS'

# Bucket logic invariants

if len(recommender['tables']['top_best_now']) == recommender['params']['top_n_best']:
    report['Recommender_tables_best_now_bucket_size'] = 'PASS'
else:
    report['Recommender_tables_best_now_bucket_size'] = 'FAIL'

if len(recommender['tables']['top_stretch']) == recommender['params']['top_n_stretch']:
    report['Recommender_tables_best_now_bucket_size'] = 'PASS'
else:
    report['Recommender_tables_best_now_bucket_size'] = 'FAIL'


best_ids = set(recommender['tables']['top_best_now'].index)

stretch_ids = set(recommender['tables']['top_stretch'].index)

overlap = best_ids & stretch_ids

if len(overlap) == 0:
    report['Recommender_tables_bucket_overlap'] = 'PASS'
else:
    report['Recommender_tables_bucket_overlap'] = 'FAIL'

if (recommender['tables']['top_best_now']['score'] == recommender['tables']['top_best_now']['score'].sort_values(ascending=False)).mean() == 1:
    report['Recommender_tables_best_now_score_order_invariant'] = 'PASS'
else:
    report['Recommender_tables_best_now_score_order_invariant'] = 'FAIL'

if (recommender['tables']['top_stretch']['score'] == recommender['tables']['top_stretch']['score'].sort_values(ascending=False)).mean() == 1:
    report['Recommender_tables_stretch_score_order_invariant'] = 'PASS'
else:
    report['Recommender_tables_stretch_score_order_invariant'] = 'FAIL'


# Stuff exist in recommender
if isinstance(explanation['tables'], dict):
    report['Explanation_tables'] = 'PASS'
else:
    report['Explanation_tables'] = 'FAIL'

if isinstance(explanation['tables']['scored_universe_explained'], pd.DataFrame):
    report['Explanation_tables_universe'] = 'PASS'
else:
    report['Explanation_tables_universe'] = 'FAIL'



required_cols = ['job_id', 'missing_families', 'n_missing_families', 'salary_context', 'why_rank', 'covered_families', 'why_bucket', 'n_covered_families']


if _require_cols(explanation['tables']['top_stretch_explained'].reset_index(), required_cols, where = 'top_stretch_explained'):
    report['Explanation_tables_stretch_cols'] = 'PASS'
else:
    report['Explanation_tables_stretch_cols'] = 'FAIL'

if _require_cols(explanation['tables']['top_best_explained'].reset_index(), required_cols, where = 'top_best_explained'):
    report['Explanation_tables_best_now_cols'] = 'PASS'
else:
    report['Explanation_tables_best_now_cols'] = 'FAIL'



if type(explanation['tables']['top_best_explained']['missing_families'][0]) == list:
    report['Explanation_tables_best_now_missing_type'] = 'PASS'
else:
    report['Explanation_tables_best_now_missing_type'] = 'FAIL'


if type(explanation['tables']['top_stretch_explained']['missing_families'][0]) == list:
    report['Explanation_tables_stretch_missing_type'] = 'PASS'
else:
    report['Explanation_tables_stretch_missing_type'] = 'FAIL'

if all(id in set(explanation['tables']['scored_universe_explained']['job_id']) for id in explanation['tables']['top_best_explained']['job_id'].to_list()):
    report['Explanation_tables_best_now_id_in_universe'] = 'PASS'
else:
    report['Explanation_tables_best_now_id_in_universe'] = 'FAIL'


if all(id in set(explanation['tables']['scored_universe_explained']['job_id']) for id in explanation['tables']['top_stretch_explained']['job_id'].to_list()):
    report['Explanation_tables_strech_in_id_universe'] = 'PASS'
else:
    report['Explanation_tables_strech_id_in_universe'] = 'FAIL'



# Stuff exist in upskilling
if isinstance(upskill, dict):
    report['Upskill_tables'] = 'PASS'
else:
    report['Upskill_tables'] = 'FAIL'

if isinstance(upskill['upskill_recommendation'], pd.DataFrame):
    report['Upskill_tables_recommendation'] = 'PASS'
else:
    report['Upskill_tables_recommendation'] = 'FAIL'

if len(upskill['upskill_recommendation']) == upskill['meta']['top_n_skills']:
    report['Upskill_tables_recommendation_topN'] = 'PASS'
else:
    report['Upskill_tables_recommendation_topN'] = 'FAIL'

if isinstance(upskill['recommendation_dict'], dict):
    report['Upskill_recom_dict'] = 'PASS'
else:
    report['Upskill_recom_dict'] = 'FAIL'

if all(upskill['scenario_meta']['status'].isin(['kept', 'skipped'])):
    report['Upskill_scenario_meta'] = 'PASS'
else:
    report['Upskill_scenario_meta'] = 'FAIL'

if isinstance(upskill['job_base_upskill'], pd.DataFrame):
    report['Upskill_tables_job_base_upskill'] = 'PASS'
else:
    report['Upskill_tables_job_base_upskill'] = 'FAIL'

if all(upskill['job_base_upskill']['upskill_scenario'].value_counts()  == len(recommender['tables']['scored_universe'])):
    report['Upskill_frozen_universe_invariant'] = 'PASS'
else:
    report['Upskill_frozen_universe_invariant'] = 'PASS'

if all(upskill['upskill_summary']['passes_guardrail']):
    report['Upskill_guardrail'] = 'PASS'
else:
    report['Upskill_guardrail'] = 'FAIL'

## Test eval script

In [12]:
from src.job_intel.evaluation.chapter4_pipeline_eval import run_ch4_pipeline_eval
summary_df, all_passed, bundle = run_ch4_pipeline_eval()

Chapter 4 pipeline eval: PASS


# === End of Notebook ===